In [1]:
import os
import argparse
import pickle
import random
import numpy as np
import logging
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from free_range_zoo.envs import wildfire_v0
from free_range_zoo.wrappers.action_task import action_mapping_wrapper_v0

In [2]:

MAX_TASKS = 100
SELF_OBS_DIM = 4
TASK_OBS_DIM = 4
INPUT_DIM = SELF_OBS_DIM + MAX_TASKS * TASK_OBS_DIM
OUTPUT_DIM = MAX_TASKS + 1

In [3]:
class QNet(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.fc1 = nn.Linear(INPUT_DIM, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.out = nn.Linear(hidden, OUTPUT_DIM)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.out(x)
    
def pad(tasks_2d: torch.Tensor, max_tasks: int) -> torch.Tensor:
    num_tasks = tasks_2d.shape[0]
    if num_tasks >= max_tasks:
        tasks_2d = tasks_2d[:max_tasks]  # truncate
    else:
        needed = max_tasks - num_tasks
        if needed > 0:
            pad = torch.zeros((needed, TASK_OBS_DIM), dtype=tasks_2d.dtype, device=tasks_2d.device)
            tasks_2d = torch.cat([tasks_2d, pad], dim=0)
    return tasks_2d

def process_env_obs(obs_dict, env_i: int) -> torch.Tensor:
    self_obs = obs_dict["self"][env_i].float()
    tasks_data = obs_dict["tasks"]
    if tasks_data.is_nested:
        tasks_2d = tasks_data[env_i].to(torch.float32)
    else:
        tasks_2d = tasks_data[env_i].to(torch.float32)

    if tasks_2d.numel() == 0:
        tasks_2d = torch.zeros((0, TASK_OBS_DIM), dtype=torch.float32, device=self_obs.device)

    tasks_fixed = pad(tasks_2d, MAX_TASKS)
    tasks_flat = tasks_fixed.flatten()
    final_obs = torch.cat([self_obs, tasks_flat], dim=0)
    return final_obs

def collect_batched_obs(obs_dict, parallel_envs: int) -> torch.Tensor:
    out_list = []
    for i in range(parallel_envs):
        vec = process_env_obs(obs_dict, i)
        out_list.append(vec)
    return torch.stack(out_list, dim=0)

In [13]:
class DQNAgent:
    def __init__(self, agent_name, lr=1e-3, gamma=0.99, epsilon_start=1.0, epsilon_end=0.1, epsilon_decay=5000, device="cpu"):
        self.agent_name = agent_name
        self.device = device
        self.gamma = gamma
        self.epsilon_start = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        self.epsilon = epsilon_start
        self.steps_done = 0
        self.q_network = QNet().to(device)
        self.target_network = QNet().to(device)
        self.target_network.load_state_dict(self.q_network.state_dict())
        self.target_network.eval()
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=lr)
        self.memory = []
        self.capacity = 20000
        self.batch_size = 64
        
    def pick_action_indices(self, obs_batch: torch.Tensor, real_task_counts: torch.Tensor) -> torch.Tensor:
        self.steps_done += 1
        fraction = min(float(self.steps_done) / self.epsilon_decay, 1.0)
        self.epsilon = self.epsilon_start + fraction * (self.epsilon_end - self.epsilon_start)
        with torch.no_grad():
            qvals = self.q_network(obs_batch)
        for i in range(obs_batch.shape[0]):
            valid_up_to = min(real_task_counts[i].item(), MAX_TASKS)
            for t in range(valid_up_to, MAX_TASKS):
                qvals[i, t] = -1e10
        if random.random() < self.epsilon:
            actions = []
            for i in range(obs_batch.shape[0]):
                valid_up_to = min(real_task_counts[i].item(), MAX_TASKS)
                candidates = list(range(valid_up_to)) + [MAX_TASKS]
                actions.append(random.choice(candidates))
            return torch.tensor(actions, device=self.device, dtype=torch.long)
        else:
            return torch.argmax(qvals, dim=1)

    def store(self, transitions):
        for tr in transitions:
            if len(self.memory) >= self.capacity:
                self.memory.pop(0)
            self.memory.append(tr)

    def update(self):
        if len(self.memory) < self.batch_size:
            return
        batch = random.sample(self.memory, self.batch_size)
        obs, action, reward, next_obs, done, real_tasks = zip(*batch)
        obs = torch.stack(obs).to(self.device)
        action = torch.tensor(action, device=self.device).long()
        reward = torch.tensor(reward, device=self.device).float()
        next_obs = torch.stack(next_obs).to(self.device)
        done = torch.tensor(done, device=self.device).float()
        real_tasks = torch.tensor(real_tasks, device=self.device).long()
        qvals = self.q_network(obs)
        chosen_q = qvals.gather(1, action.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            nxt_qvals = self.target_network(next_obs)
            for i in range(self.batch_size):
                valid_up_to = min(real_tasks[i].item(), MAX_TASKS)
                for t in range(valid_up_to, MAX_TASKS):
                    nxt_qvals[i, t] = -1e10
            max_next_q = torch.max(nxt_qvals, dim=1)[0]
            target_q = reward + (1 - done) * self.gamma * max_next_q
        loss = F.mse_loss(chosen_q, target_q)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        if self.steps_done % 200 == 0:
            self.target_network.load_state_dict(self.q_network.state_dict())

    def save(self, path):
        torch.save({
            'q_network': self.q_network.state_dict(),
            'target_network': self.target_network.state_dict()
        }, path)

    def load(self, path):
        ckp = torch.load(path, map_location=self.device)
        self.q_network.load_state_dict(ckp['q_network'])
        self.target_network.load_state_dict(ckp['target_network'])
        

In [14]:
def train(config, output, episodes, steps, parallel_envs, cuda):
    os.makedirs(output, exist_ok=True)
    logging.basicConfig(level=logging.INFO)
    logger = logging.getLogger("train")
    device = torch.device("cuda" if cuda and torch.cuda.is_available() else "cpu")
    random.seed(0)
    np.random.seed(0)
    torch.manual_seed(0)
    with open(config, "rb") as f:
        configuration = pickle.load(f)
    env = wildfire_v0.parallel_env(
        parallel_envs=parallel_envs,
        max_steps=steps,
        configuration=configuration,
        device=device,
        show_bad_actions=True,
        observe_other_power=True,
        observe_other_suppressant=True,
        buffer_size=parallel_envs * steps,
    )
    env = action_mapping_wrapper_v0(env)
    agent_names = env.agents
    logger.info(f"Agents: {agent_names}")
    agents = {ag_name: DQNAgent(ag_name, device=device) for ag_name in agent_names}
    logger.info("Starting training...")
    for ep in range(episodes):
        obs, infos = env.reset()
        done_tensor = torch.all(env.finished, dim=0)
        done = done_tensor.all().item()
        ep_rewards = {ag: 0.0 for ag in agent_names}
        step_count = 0
        while not done:
            actions_dict = {}
            for ag_name in agent_names:
                obs_dict, _ = obs[ag_name]
                batched_obs = collect_batched_obs(obs_dict, parallel_envs).to(device)
                real_task_counts = []
                tasks_data = obs_dict["tasks"]
                if tasks_data.is_nested:
                    for i in range(parallel_envs):
                        real_task_counts.append(tasks_data[i].shape[0])
                else:
                    for i in range(parallel_envs):
                        real_task_counts.append(tasks_data[i].shape[0])
                real_task_counts = torch.tensor(real_task_counts, device=device)
                action_indices = agents[ag_name].pick_action_indices(batched_obs, real_task_counts)
                action_tensors = []
                for env_i in range(parallel_envs):
                    idx = action_indices[env_i].item()
                    rt = real_task_counts[env_i].item()
                    if idx < rt and idx < MAX_TASKS:
                        action_tensors.append([idx, 0])
                    else:
                        action_tensors.append([0, -1])
                actions_dict[ag_name] = torch.tensor(action_tensors, dtype=torch.int32, device=device)
            next_obs, rewards, terms, truncs, infos = env.step(actions_dict)
            for ag_name in agent_names:
                obs_dict, _ = obs[ag_name]
                batched_current = collect_batched_obs(obs_dict, parallel_envs)
                next_dict, _ = next_obs[ag_name]
                batched_next = collect_batched_obs(next_dict, parallel_envs)
                tasks_data_cur = obs_dict["tasks"]
                real_tasks_list = []
                if tasks_data_cur.is_nested:
                    for i in range(parallel_envs):
                        real_tasks_list.append(tasks_data_cur[i].shape[0])
                else:
                    for i in range(parallel_envs):
                        real_tasks_list.append(tasks_data_cur[i].shape[0])
                action_array = actions_dict[ag_name]
                final_action_idxs = []
                for i in range(parallel_envs):
                    (task_idx, sub_a) = action_array[i].tolist()
                    rt = real_tasks_list[i]
                    if sub_a == 0:
                        final_action_idxs.append(task_idx)
                    else:
                        final_action_idxs.append(MAX_TASKS)
                r = rewards[ag_name].float()
                d = (terms[ag_name] | truncs[ag_name]).float()
                transitions = []
                for i in range(parallel_envs):
                    transitions.append((
                        batched_current[i],
                        final_action_idxs[i],
                        r[i].item(),
                        batched_next[i],
                        d[i].item(),
                        real_tasks_list[i]
                    ))
                agents[ag_name].store(transitions)
                ep_rewards[ag_name] += r.sum().item()
            done_tensor = torch.all(env.finished, dim=0)
            done = done_tensor.all().item()
            step_count += 1
            for ag_name in agent_names:
                agents[ag_name].update()
            obs = next_obs
        logger.info(f"Episode {ep + 1}/{episodes} done in {step_count} steps. Rewards={ep_rewards}")
    for ag_name, ag in agents.items():
        path = os.path.join(output, f"{ag_name}_dqn.pt")
        ag.save(path)
        logger.info(f"Saved {ag_name} to {path}")
    env.close()

In [ ]:
if __name__ == "__main__":
    config = "moasei_rl/configs/WS1.pkl"
    output = "moasei_rl/output"
    episodes = 10000
    steps = 100
    parallel_envs = 50
    cuda = True
    
    
    train(config, output, episodes, steps, parallel_envs, cuda)